# S3 — Pandas 轉換：groupby / merge / pivot

> **時間**：2 小時  
> **資料**：`orders_clean.csv`（S2 產出）+ `customers.csv` + `products.csv`  
> **學完能幹嘛**：用 SQL 的思維在 Pandas 中做三表 join、分組統計、樞紐分析

---

## 上節回顧 + 本節為什麼重要

S2 我們把髒訂單清乾淨了，存成 `orders_clean.csv`。但單看一張訂單表只看得到數字，看不到**商業洞察**，例如：

- 哪個**地區**的客人買最多？ → 需要把 orders 和 customers join
- 哪個**商品類別**貢獻最多營收？ → 需要把 orders 和 products join
- 每個地區的 Top 3 熱銷商品？ → 需要 groupby + 排序

> **DE/DA 視角**：如果你會 SQL，本節你會很爽；如果不會，本節學完你就等於會了 80% 的 SQL。

**三大工具**：
1. **`merge`** = SQL 的 JOIN（合併多表）
2. **`groupby`** = SQL 的 GROUP BY（分組統計）
3. **`pivot_table`** = Excel 的樞紐分析表（二維交叉統計）


In [29]:
import pandas as pd
import numpy as np

DATA = '../datasets/ecommerce' #共同路徑

orders    = pd.read_csv(f'{DATA}/orders_clean.csv', parse_dates=['order_date'])
customers = pd.read_csv(f'{DATA}/customers.csv')
products  = pd.read_csv(f'{DATA}/products.csv')

print('orders   :', orders.shape,    list(orders.columns))
print('customers:', customers.shape, list(customers.columns))
print('products :', products.shape,  list(products.columns))

orders   : (188, 6) ['order_id', 'customer_id', 'product_id', 'qty', 'order_date', 'amount']
customers: (26, 5) ['customer_id', 'customer_name', 'region', 'signup_date', 'vip_level']
products : (30, 5) ['product_id', 'product_name', 'category', 'unit_price', 'stock_qty']


In [30]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from common.checker import check

---
## 核心觀念 1／3 — 條件篩選（複習 + 加強）

S2 學過 `df.loc[mask]`，本節加強兩招：`isin` 和 `between`。


In [45]:
# 1) 找出金額在 1000~5000 之間的訂單
orders['amount'] = pd.to_numeric(orders['amount'], errors='coerce')
mid = orders[orders['amount'].between(1000, 5000)]
print('中間金額訂單:', len(mid))

中間金額訂單: 81


In [44]:
orders['amount'] = pd.to_numeric(orders['amount'], errors='coerce')

mid = orders[(orders['amount'] >= 1000) & (orders['amount'] <= 5000)]
print('中間金額訂單:', len(mid))   

中間金額訂單: 81


In [ ]:
print(orders.dtypes)

order_id                int64
customer_id             int64
product_id              int64
qty                   float64
order_date     datetime64[us]
amount                    str
dtype: object


In [ ]:
# 1) 找出金額在 1000~5000 之間的訂單
mid = orders[orders['amount'].between(1000, 5000)]
print('中間金額訂單:', len(mid))

# 2) 找出指定顧客的訂單（用 isin 一次篩多個值）
vip_ids = [2001, 2005, 2010]
vip_orders = orders[orders['customer_id'].isin(vip_ids)]
print('VIP 訂單:', len(vip_orders))

# 3) 複合條件：高金額 + 特定顧客
hit = orders[(orders['amount'] > 3000) & (orders['customer_id'].isin(vip_ids))]
print('複合條件命中:', len(hit))

中間金額訂單: 81
VIP 訂單: 19
複合條件命中: 9


**口訣**：`between` 找區間、`isin` 找清單、`&/|` 組合條件（記得加括號）。


---
## 核心觀念 2／3 — `groupby`：分組 + 聚合

**SQL 對照**：
```sql
SELECT customer_id, SUM(amount) 
FROM orders 
GROUP BY customer_id;
```
↓ 轉成 Pandas ↓
```python
orders.groupby('customer_id')['amount'].sum()
```


In [ ]:
# 1) 最簡單：每個顧客的總消費
per_customer = orders.groupby('customer_id')['amount'].sum()
print(per_customer.head())
print('型別:', type(per_customer).__name__)  # Series

customer_id
2001     6865.0
2002    12278.0
2003    22533.0
2004    33345.0
2005    26569.0
Name: amount, dtype: float64
型別: Series


In [ ]:
# 2) 多種聚合一次做：count, sum, mean
summary = orders.groupby('customer_id')['amount'].agg(['count', 'sum', 'mean'])
summary.columns = ['訂單數', '總金額', '平均金額']
print(summary.head())

# 3) 排序找 Top 5
print('\n💰 總金額 Top 5 顧客:')
print(summary.sort_values('總金額', ascending=False).head())

             訂單數      總金額         平均金額
customer_id                           
2001           2   6865.0  3432.500000
2002           5  12278.0  2455.600000
2003           9  22533.0  2503.666667
2004           5  33345.0  6669.000000
2005           7  26569.0  3795.571429

💰 總金額 Top 5 顧客:
             訂單數      總金額         平均金額
customer_id                           
2022          13  51242.0  3941.692308
2015          11  46790.0  4253.636364
2025          10  40128.0  4012.800000
2021           8  38430.0  4803.750000
2018          10  35856.0  3585.600000


In [ ]:
# 4) 多欄聚合：不同欄用不同函式
multi = orders.groupby('customer_id').agg(
    訂單數=('order_id', 'count'),
    總金額=('amount', 'sum'),
    平均金額=('amount', 'mean'),
    最大單筆=('amount', 'max'),
).reset_index()
multi.head()

,customer_id,訂單數,總金額,平均金額,最大單筆
0,2001,3,6865.0,3432.500000,5625.0
1,2002,7,12278.0,2455.600000,6472.0
2,2003,10,22533.0,2503.666667,7076.0
3,2004,7,33345.0,6669.000000,10750.0
4,2005,8,26569.0,3795.571429,10750.0


**口訣**：`groupby(分組欄).agg(...)` → 一行程式替代 SQL 的整段 GROUP BY。


> ### 💡 知識補給站 — groupby 背後的 Hash Table
> 
> `groupby('customer_id')` 底層做了什麼？Pandas 建了一個 **hash table**（雜湊表），把每個 `customer_id` 映射到它出現的**行號清單**，然後對每組套用聚合函式。
> 
> - 這跟 SQL 的 `GROUP BY` 概念一模一樣 — 資料庫也是用 **hash aggregate** 或 sort aggregate
> - Hash table 的查找效率是 **O(1)**，所以 groupby 整體效能是 **O(n)** — 100 萬筆訂單也能在毫秒內完成
> - 效能瓶頸不在分組數量，而在每組的聚合運算複雜度
> 
> 這也是為什麼 Pandas 的 `groupby` 跟用 `for` 迴圈自己分組比起來快非常多 — 底層是 C 語言實作的 hash table，不是 Python 層級的字典操作。
> 
> *延伸關鍵字：hash table, hash aggregate, O(n) complexity, data structure*

---
## 核心觀念 3／3 — `merge`：合併多表（= SQL JOIN）

四種 join 一張圖記熟：

| `how=` | 意義 | 什麼時候用 |
|---|---|---|
| `'inner'` | 兩邊都有才保留 | 最安全、最常用 |
| `'left'`  | 保留左表全部 | 以主表為準 |
| `'right'` | 保留右表全部 | 少用 |
| `'outer'` | 全部都保留 | 找遺漏用 |


In [34]:
orders.head()

,order_id,customer_id,product_id,qty,order_date,amount
0,5064,2022,1026,4.0,2025-03-26,8600
1,5023,2026,1021,5.0,2025-01-05,1355
2,5123,2013,1013,2.0,2025-09-11,"$3,538"
3,5118,2005,1028,1.0,2025-05-22,1618
4,5161,2017,1019,3.0,2025-08-20,1846


In [33]:
customers.head()

,customer_id,customer_name,region,signup_date,vip_level
0,2001,Alice Chen,South,2023-10-01,Silver
1,2002,Bob Wang,West,2023-07-14,Gold
2,2003,Carol Huang,South,2023-12-17,Gold
3,2004,David Chen,South,2024-02-25,Bronze
4,2005,Emma Liu,West,2023-05-18,Bronze


In [35]:
# 1) 兩表 join：orders + customers（看每筆訂單屬於哪個地區）
oc = orders.merge(customers, on='customer_id', how='left')
print('合併後形狀:', oc.shape)
print('新增的欄位:', [c for c in oc.columns if c not in orders.columns])
oc.head()

合併後形狀: (188, 10)
新增的欄位: ['customer_name', 'region', 'signup_date', 'vip_level']


,order_id,customer_id,product_id,qty,order_date,amount,customer_name,region,signup_date,vip_level
0,5064,2022,1026,4.0,2025-03-26,8600,Victor Lin,North,2023-02-27,Gold
1,5023,2026,1021,5.0,2025-01-05,1355,Zoe Huang,South,2023-05-16,Platinum
2,5123,2013,1013,2.0,2025-09-11,"$3,538",Mia Huang,North,2023-07-17,Platinum
3,5118,2005,1028,1.0,2025-05-22,1618,Emma Liu,West,2023-05-18,Bronze
4,5161,2017,1019,3.0,2025-08-20,1846,Quinn Chen,East,2023-08-11,Silver


In [36]:
# 2) 三表 join：再加上 products
#    注意：orders 裡是 product_id，products 裡也是 product_id → on='product_id'
full = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)
print('三表合併後形狀:', full.shape)
print('欄位:', list(full.columns))
full.head()

三表合併後形狀: (188, 14)
欄位: ['order_id', 'customer_id', 'product_id', 'qty', 'order_date', 'amount', 'customer_name', 'region', 'signup_date', 'vip_level', 'product_name', 'category', 'unit_price', 'stock_qty']


,order_id,customer_id,product_id,qty,order_date,amount,customer_name,region,signup_date,vip_level,product_name,category,unit_price,stock_qty
0,5064,2022,1026,4.0,2025-03-26,8600,Victor Lin,North,2023-02-27,Gold,Dumbbell 5kg,Sports,2150,51
1,5023,2026,1021,5.0,2025-01-05,1355,Zoe Huang,South,2023-05-16,Platinum,Throw Pillow,Home,271,150
2,5123,2013,1013,2.0,2025-09-11,"$3,538",Mia Huang,North,2023-07-17,Platinum,Cotton T-Shirt,Clothing,1769,174
3,5118,2005,1028,1.0,2025-05-22,1618,Emma Liu,West,2023-05-18,Bronze,Water Bottle,Sports,1618,186
4,5161,2017,1019,3.0,2025-08-20,1846,Quinn Chen,East,2023-08-11,Silver,Coffee Mug,Home,1846,274


**口訣**：`merge(另一張表, on='共同欄', how='left/inner')` — 從左表的角度去黏上右表的資訊。


> ### 💡 知識補給站 — merge 的隱藏陷阱：validate 參數
> 
> `merge` 預設**不檢查 key 的唯一性**。如果 `products.csv` 裡同一個 `product_id` 出現兩次，merge 後列數會**暴增**（笛卡爾積），而且**不會報錯** — 這是真實工作中最常見的 silent data quality bug。
> 
> 解法：加上 `validate` 參數，key 不符預期會直接噴 `MergeError`：
> ```python
> orders.merge(products, on='product_id', how='left', validate='many_to_one')
> ```
> 
> | validate 值 | 意義 |
> |---|---|
> | `'one_to_one'` | 兩邊的 key 都必須唯一 |
> | `'many_to_one'` | 右表的 key 必須唯一（最常用） |
> | `'one_to_many'` | 左表的 key 必須唯一 |
> 
> 類比：結婚登記處會檢查「雙方都是未婚」— 不檢查就會出問題。
> 
> **實務建議**：每次 merge 完第一件事就是 `print(df.shape)` 確認列數是否合理。
> 
> *延伸關鍵字：merge validate, Cartesian product, data quality, join explosion*

---
## 實務 Case — 電商銷售洞察分析

**情境**：老闆下午開會要你回答三個問題：
1. 🏆 各**地區**的總營收排名？
2. 🔥 各地區的 **Top 3 熱銷商品**是什麼？
3. 💎 **VIP 等級**的營收貢獻佔比？


In [41]:
# 先把三表合併起來當分析基底
df = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)
print('分析基底:', df.shape)
df[['order_id','customer_name','region','vip_level','product_name','category','amount']].head()

分析基底: (188, 14)


,order_id,customer_name,region,vip_level,product_name,category,amount
0,5064,Victor Lin,North,Gold,Dumbbell 5kg,Sports,8600
1,5023,Zoe Huang,South,Platinum,Throw Pillow,Home,1355
2,5123,Mia Huang,North,Platinum,Cotton T-Shirt,Clothing,"$3,538"
3,5118,Emma Liu,West,Bronze,Water Bottle,Sports,1618
4,5161,Quinn Chen,East,Silver,Coffee Mug,Home,1846


In [48]:
# Q1: 各地區總營收排名
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
region_rev = df.groupby('region')['amount'].sum().sort_values(ascending=False)
print('🏆 地區營收排名:')
print(region_rev)

🏆 地區營收排名:
region
South    233089.0
North    222365.0
West      96140.0
East      43362.0
Name: amount, dtype: float64


In [49]:
# Q2: 各地區 Top 3 熱銷商品（by 銷售金額）
#     技巧：先 groupby 兩層 → sort → 每個地區取前 3
product_by_region = (
    df.groupby(['region', 'product_name'])['amount']
      .sum()
      .reset_index()
      .sort_values(['region', 'amount'], ascending=[True, False])
)

top3 = product_by_region.groupby('region').head(3)
print('🔥 各地區 Top 3 熱銷商品:')
print(top3)

🔥 各地區 Top 3 熱銷商品:
   region    product_name   amount
2    East      Clean Code   9580.0
1    East      Candle Set   8348.0
12   East    Water Bottle   6472.0
17  North      Clean Code  28740.0
35  North  Statistics 101  24096.0
37  North    Water Bottle  19416.0
65  South        Yoga Mat  31216.0
62  South    Water Bottle  21034.0
44  South  Cotton T-Shirt  19459.0
71   West    Dumbbell 5kg  15050.0
66   West      Candle Set  14609.0
75   West       ML Primer  10950.0


In [50]:
# Q3: VIP 等級貢獻佔比（用 pivot_table 或 groupby 都可以）
vip_rev = df.groupby('vip_level')['amount'].sum()
vip_pct = (vip_rev / vip_rev.sum() * 100).round(1)

vip_report = pd.DataFrame({'總金額': vip_rev, '佔比(%)': vip_pct})
vip_report = vip_report.sort_values('總金額', ascending=False)
print('💎 VIP 等級貢獻:')
print(vip_report)

💎 VIP 等級貢獻:
                總金額  佔比(%)
vip_level                 
Gold       256321.0   43.1
Platinum   146119.0   24.6
Bronze     126193.0   21.2
Silver      66323.0   11.1


In [51]:
# Bonus: pivot_table — 交叉看「地區 × 品類」的營收矩陣
pivot = df.pivot_table(
    index='region',
    columns='category',
    values='amount',
    aggfunc='sum',
    fill_value=0,
)
print('📊 地區 × 品類 營收交叉表:')
print(pivot)

📊 地區 × 品類 營收交叉表:
category    Books  Clothing  Electronics     Home   Sports
region                                                    
East       9580.0    2396.0       5311.0  12153.0  13922.0
North     80292.0   39852.0      24837.0  30919.0  46465.0
South     49404.0   55277.0      31718.0  11681.0  85009.0
West      24124.0   15352.0      13283.0  15884.0  27497.0


In [ ]:
# 存一份合併好的分析基底給 S4/S5/S6 用
df.to_csv('../datasets/ecommerce/orders_enriched.csv', index=False)
print('已存 orders_enriched.csv，供後續 session 使用')

---
## 課堂練習（40 min）

### 🟢 送分題
用 `orders` 算出：
1. 每個 `customer_id` 的訂單總數
2. 平均單筆訂單金額


In [66]:
# TODO
import pandas as pd
import numpy as np

DATA = '../datasets/ecommerce' #共同路徑

orders    = pd.read_csv(f'{DATA}/orders_clean.csv', parse_dates=['order_date'])
orders['amount'] = pd.to_numeric(orders['amount'], errors='coerce')

summary = orders.groupby('customer_id')['amount'].agg(['count', 'mean'])
summary.columns = ['訂單數', '平均單筆金額']
print(summary)

             訂單數       平均單筆金額
customer_id                  
2001           2  3432.500000
2002           5  2455.600000
2003           9  2503.666667
2004           5  6669.000000
2005           7  3795.571429
2006           1  2087.000000
2007           5  3202.000000
2008           4  7267.500000
2009           7  3582.714286
2010           8  3810.125000
2011           8  2064.500000
2012           3  1696.666667
2013           4  3376.750000
2014           5  3240.800000
2015          11  4253.636364
2016           5  3292.800000
2017           7  3544.428571
2018          10  3585.600000
2019           2  1486.500000
2020           5  3646.800000
2021           8  4803.750000
2022          13  3941.692308
2023           8  4330.875000
2024           7  2860.571429
2025          10  4012.800000
2026           5  1944.600000


In [ ]:
import pandas as pd
import numpy as np

DATA = '../datasets/ecommerce' #共同路徑

orders    = pd.read_csv(f'{DATA}/orders_clean.csv', parse_dates=['order_date'])
customers = pd.read_csv(f'{DATA}/customers.csv')
products  = pd.read_csv(f'{DATA}/products.csv')

print('orders   :', orders.shape,    list(orders.columns))
print('customers:', customers.shape, list(customers.columns))
print('products :', products.shape,  list(products.columns))

In [72]:
df = orders.merge(products, on='product_id' , how='left')

df.head()

#先 qty * unit_price = amount 算出每筆訂單的總金額 (如果 orders 裡沒有 amount 的話))
#每筆訂單的金額 / 總共訂單的數量 = 平均每筆訂單金額

orders["amount"].sum() / orders["order_id"].count()

np.float64(3164.659574468085)

### 🟡 核心題
用三表合併的 `df`，回答：
1. 哪個**商品類別**總營收最高？
2. **Gold** 等級的顧客一共下了多少筆訂單？金額多少？
3. 各**地區**的平均訂單金額是多少？


In [74]:
# TODO
df = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)
print('分析基底:', df.shape)
df[['order_id','customer_name','region','vip_level','product_name','category','amount']].head()




分析基底: (188, 14)


,order_id,customer_name,region,vip_level,product_name,category,amount
0,5064,Victor Lin,North,Gold,Dumbbell 5kg,Sports,8600.0
1,5023,Zoe Huang,South,Platinum,Throw Pillow,Home,1355.0
2,5123,Mia Huang,North,Platinum,Cotton T-Shirt,Clothing,NaN
3,5118,Emma Liu,West,Bronze,Water Bottle,Sports,1618.0
4,5161,Quinn Chen,East,Silver,Coffee Mug,Home,1846.0


In [75]:
df.groupby("category")["amount"].sum().sort_values(ascending=False)

category
Sports         172893.0
Books          163400.0
Clothing       112877.0
Electronics     75149.0
Home            70637.0
Name: amount, dtype: float64

In [77]:
Gold_amount = df[df["vip_level"] == "Gold"]["amount"].count()
Gold_total = df[df["vip_level"] == "Gold"]["amount"].sum()

summary_df = pd.DataFrame({
    "VIP 等級": ["Gold"],
    "訂單數量": [Gold_amount],
    "總金額": [Gold_total]
})

summary_df

,VIP 等級,訂單數量,總金額
0,Gold,69,256321.0


In [79]:
(df.groupby("region")["amount"].sum() / df.groupby("region")["order_id"].count()).round(1)

region
East     2710.1
North    3318.9
South    3237.3
West     2913.3
dtype: float64

### 🔴 挑戰題
計算每位顧客的 **RFM 粗估**：
- **R**ecency：最近一次下單日期（用 `max(order_date)`）
- **F**requency：下單次數
- **M**onetary：總金額

最後用 M 排序，列出 Top 5 顧客（加上顧客名字）。

> 這是電商最經典的顧客分群指標，會做這題你就有 junior DA 實力了。


In [81]:
# TODO

import time
import pandas as pd
from datetime import datetime

now = datetime.now()
print(now)

2026-04-26 11:39:13.264987


In [ ]:
import time

df["Recency"] = (now - df["order_date"]) #最近購買
df["Frequency"] = df.groupby("customer_id")["order_id"].count() #購買頻率
df["Monetary"] = df.groupby("customer_id")["amount"].sum() 



0     396 days 11:39:13.264987
1     476 days 11:39:13.264987
2     227 days 11:39:13.264987
3     339 days 11:39:13.264987
4     249 days 11:39:13.264987
                ...           
183   437 days 11:39:13.264987
184   205 days 11:39:13.264987
185   479 days 11:39:13.264987
186   327 days 11:39:13.264987
187   233 days 11:39:13.264987
Name: order_date, Length: 188, dtype: timedelta64[us]

In [83]:
# 🏁 大地遊戲 — 哪個品類總營收最高？答對就能找到解答！
check('S3', top_category)

NameError: name 'top_category' is not defined

---
## 小結與速查表

| 想做什麼 | 怎麼寫 | SQL 對照 |
|---|---|---|
| 區間篩選 | `df[df['x'].between(a,b)]` | `WHERE x BETWEEN a AND b` |
| 清單篩選 | `df[df['x'].isin([...])]` | `WHERE x IN (...)` |
| 分組加總 | `df.groupby('k')['v'].sum()` | `GROUP BY k` |
| 多聚合 | `df.groupby('k').agg(...)` | 多 `SUM/COUNT/AVG` |
| Join 兩表 | `a.merge(b, on='k', how='left')` | `LEFT JOIN` |
| 樞紐分析 | `df.pivot_table(index, columns, values)` | (Excel 樞紐) |
| 每組 Top N | `sort + groupby.head(n)` | `ROW_NUMBER()` |

**下節預告 S4**：我們有 `order_date`，但還沒用它做時序分析。下節會教你用 1 行程式算出「月度營收」「7 日移動平均」「年月週趨勢」。
